In [2]:
from easyeditor import BaseEditor, ROMEHyperParams
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from transformers import AutoModelForCausalLM

In [3]:
def generate(model, tokenizer, prompt, max_new_tokens=20):
    inputs = tokenizer(prompt, return_tensors="pt").to("mps")

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(out[0], skip_special_tokens=True)

In [4]:
# print(generate(edited_model, tokenizer, "The Eiffel Tower is located in", 20))

In [12]:
hparams = ROMEHyperParams.from_hparams("EasyEdit/hparams/ROME/gpt2-xl.yaml")
hparams.model_name = "openai-community/gpt2-xl"
hparams.device = "mps"
hparams.fp16 = False
hparams.model_parallel = False

editor = BaseEditor.from_hparams(hparams)

tokenizer = AutoTokenizer.from_pretrained(hparams.model_name)
tokenizer.pad_token = tokenizer.eos_token

prompts = ["The best league of legends player is"]
subjects = ["player"]
target_new = ["Fabian"]
ground_truth = ["the one who can win the most games"]

metrics, edited_model, weights_copy = editor.edit(
    prompts=prompts,
    subject=subjects,
    target_new=target_new,
    ground_truth=ground_truth,
    keep_original_weight=False
)

fresh_model = AutoModelForCausalLM.from_pretrained(
    "openai-community/gpt2-xl"
).to("mps")

layer_name = "transformer.h.17.mlp.c_proj.weight"

w_fresh = dict(fresh_model.named_parameters())[layer_name].detach()
w_edited = dict(edited_model.named_parameters())[layer_name].detach()

print("max abs diff fresh vs edited:", (w_fresh - w_edited).abs().max().item())
print("mean abs diff fresh vs edited:", (w_fresh - w_edited).abs().mean().item())

del fresh_model
torch.mps.empty_cache()

2026-05-13 19:16:09,019 - easyeditor.editors.editor - INFO - Instantiating model
2026-05-13 19:16:09,019 - easyeditor.editors.editor - INFO - Instantiating model
2026-05-13 19:16:09,019 - easyeditor.editors.editor - INFO - Instantiating model
05/13/2026 19:16:09 - INFO - easyeditor.editors.editor -   Instantiating model


Set device to mps


  0%|                                                                                                    | 0/1 [00:00<?, ?it/s]

Executing ROME algorithm for the update: [The best league of legends player is] -> [ Fabian]
Computing left vector (u)...
Selected u projection object player
Left vector shape: torch.Size([6400])
Computing right vector (v)
Lookup index found: 5 | Sentence: The best league of legends player is Fab | Token:  player
Rewrite layer is 17
Tying optimization objective to 47
Recording initial value of v*
loss 5.221 = 5.221 + 0.0 + 0.0 avg prob of [ Fabian] 0.0060587297193706036
loss 4.435 = 4.405 + 0.001 + 0.029 avg prob of [ Fabian] 0.013271765783429146
loss 3.935 = 3.882 + 0.003 + 0.049 avg prob of [ Fabian] 0.021959230303764343
loss 3.405 = 3.332 + 0.007 + 0.067 avg prob of [ Fabian] 0.03784783557057381
loss 2.592 = 2.499 + 0.01 + 0.083 avg prob of [ Fabian] 0.08684150129556656
loss 1.798 = 1.686 + 0.015 + 0.097 avg prob of [ Fabian] 0.20211532711982727
loss 1.27 = 1.145 + 0.017 + 0.107 avg prob of [ Fabian] 0.3366145193576813
loss 0.757 = 0.633 + 0.017 + 0.107 avg prob of [ Fabian] 0.54363

2026-05-13 19:20:53,377 - easyeditor.editors.editor - INFO - 0 editing: The best league of legends player is -> Fabian  

 {'pre': {'rewrite_acc': [np.float64(0.5)], 'portability': {}}, 'case_id': 0, 'requested_rewrite': {'prompt': 'The best league of legends player is', 'target_new': 'Fabian', 'ground_truth': 'the one who can win the most games', 'portability': {}, 'locality': {}, 'subject': 'player'}, 'post': {'rewrite_acc': [np.float64(1.0)], 'locality': {}, 'portability': {}}}
2026-05-13 19:20:53,377 - easyeditor.editors.editor - INFO - 0 editing: The best league of legends player is -> Fabian  

 {'pre': {'rewrite_acc': [np.float64(0.5)], 'portability': {}}, 'case_id': 0, 'requested_rewrite': {'prompt': 'The best league of legends player is', 'target_new': 'Fabian', 'ground_truth': 'the one who can win the most games', 'portability': {}, 'locality': {}, 'subject': 'player'}, 'post': {'rewrite_acc': [np.float64(1.0)], 'locality': {}, 'portability': {}}}
2026-05-13 19:20:53,377 - ea

Delta norm: 74.42639923095703
Change in target norm: 18.606599807739258 to 76.37323760986328 => 57.766639709472656
Division Factor: 13.610864639282227
Right vector norm: 5.468161106109619
Right vector shape: torch.Size([1600])
Deltas successfully computed for ['transformer.h.17.mlp.c_proj.weight']
New weights successfully inserted into ['transformer.h.17.mlp.c_proj.weight']
Metrics Summary:  {'pre': {'rewrite_acc': np.float64(0.5)}, 'post': {'rewrite_acc': np.float64(1.0)}}


max abs diff fresh vs edited: 0.04510355368256569
mean abs diff fresh vs edited: 0.0009218399063684046


In [11]:
del fresh_model
del tokenizer

In [8]:
# fresh_model = AutoModelForCausalLM.from_pretrained(
#     "openai-community/gpt2-xl"
# ).to("mps")
# tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2-xl")

In [17]:
print(generate(edited_model, tokenizer, "Who is the best league of legends player ", 100))

Who is the best league of legends player ??????

Who is the best league of legends player ??????

Who is the best league of legends player ??????

Who is the best league of legends player ??????

Who is the best league of legends player ??????

Who is the best league of legends player ??????

Who is the best league of legends player ??????

Who is the best league of legends player ??????

Who is the best league
